# Resource-Efficient CSI Prediction: GRU-Attn-DSLH vs Baselines

This notebook implements the "Resource-Efficient CSI Prediction" models described in the paper. 


In [ ]:
# Reproducibility & Setup
import os
import random
import math
import time
import json
import warnings
import gc
import platform
import copy
from dataclasses import dataclass
from typing import Tuple, List, Optional, Union
from bisect import bisect_right

import numpy as np
import h5py
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Windows-safe multiprocessing
try:
    if platform.system() == 'Windows':
        from torch.multiprocessing import set_start_method
        set_start_method('spawn', force=True)
except RuntimeError:
    pass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

def set_global_seed(seed: int):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

DEFAULT_SEED = 43
set_global_seed(DEFAULT_SEED)
print(f"Global seed set to {DEFAULT_SEED}")

In [ ]:
@dataclass
class Cfg:
    # --- Data Config ---
    # Update this path to your local dataset
    mat_path: str = r"C:\Users\MOHUSSAIN25\Downloads\dataset_train_mixed.mat" 
    out_dir: str = 'trained_streamlined'
    use_speed_feature: bool = True
    norm_per_run: bool = True
    norm_fit_train_only: bool = False
    safety_gap: bool = True  # Enforce gap between train/test partitions
    
    # --- Sequence Config ---
    npast: int = 128
    nfuture: int = 8
    train_frac: float = 0.8
    stride: int = 1
    
    # --- Model Dimensions ---
    d_model: int = 256
    d_model_linformer: int = 256
    d_model_rnn: int = 256
    n_layers: int = 3
    ffn_mult: int = 4
    dropout: float = 0.3
    
    # --- Training Config ---
    batch_size: int = 256
    epochs: int = 100
    lr: float = 3e-4
    weight_decay: float = 0.0001
    num_workers: int = 0
    
    # --- Proposed Model Specifics ---
    attn_scale: bool = True
    attn_dropout: float = 0.0
    query_proj: bool = False  # Disabled to save ~0.065M params
    gru_residual: bool = False
    gated_fusion: bool = True # Key feature: Gated Fusion
    stacked_gru: bool = False
    bidirectional: bool = False

    # --- Other ---
    use_ema: bool = False
    ema_decay: float = 0.999
    loss_type: str = 'wmse' 
    pred_l2_lambda: float = 0.0
    seed: int = 23
    
    # Early Stopping
    early_stopping: bool = True
    early_patience: int = 15
    early_min_delta: float = 1e-4

cfg = Cfg()
os.makedirs(cfg.out_dir, exist_ok=True)
print(cfg)

In [ ]:
# --- Data Loading Helpers ---

def _parse_complex(arr):
    """
    MATLAB -v7.3 saves complex numbers as a compound type with fields 'r' and 'i'
    or 'real' and 'imag'. This converts them to numpy complex64/128.
    """
    if arr.dtype.names is not None:
        if 'r' in arr.dtype.names and 'i' in arr.dtype.names:
            return arr['r'] + 1j * arr['i']
        elif 'real' in arr.dtype.names and 'imag' in arr.dtype.names:
            return arr['real'] + 1j * arr['imag']
    return arr

def _load_h5_cell_array(f, dataset_name):
    """
    Loads a MATLAB cell array from an HDF5 file.
    In v7.3, cell arrays are datasets of object references.
    """
    if dataset_name not in f:
        raise KeyError(f"Dataset '{dataset_name}' not found in HDF5 file.")

    ds = f[dataset_name]
    data_list = []
    refs = np.array(ds).flatten() # Flatten (N,1) or (1,N)

    for ref in refs:
        try:
            item = f[ref]
            vals = np.array(item)
            vals = _parse_complex(vals) 
            data_list.append(vals)
        except Exception as e:
            warnings.warn(f"Could not dereference item in {dataset_name}: {e}")

    return data_list

def _to_H_time_major(ci_coeff: np.ndarray, time_len=None) -> np.ndarray:
    """
    Convert QuaDRiGa CSI [Time, Paths, Tx, Rx] -> [Time, Features].
    (h5py reads MATLAB [Rx, Tx, Paths, Time] as [Time, Paths, Tx, Rx])
    """
    X = _parse_complex(ci_coeff)
    if X.ndim != 4:
        raise ValueError(f"Expected 4 dims [Time, Paths, Tx, Rx], got {X.shape}")

    T, P, Tx, Rx = X.shape
    H = X.sum(axis=1) # Sum over paths -> [T, Tx, Rx]
    H = H.reshape(T, -1) # Flatten spatial -> [T, Tx*Rx]
    
    # Concatenate Real/Imag -> [T, 2*Tx*Rx]
    H_feat = np.concatenate([H.real, H.imag], axis=1).astype(np.float32)
    return H_feat


class QuaDRiGaRunWindows(Dataset):
    def __init__(self, seq_pairs, npast: int, nfuture: int, stride: int = 1):
        self.npast = npast
        self.nfuture = nfuture
        self.stride = stride
        self.blocks = []
        self.cum_counts = []
        self.feature_dim = None
        self.target_dim = None
        total = 0

        for X, Y in seq_pairs:
            X = np.asarray(X, dtype=np.float32)
            Y = np.asarray(Y, dtype=np.float32)
            N = X.shape[0]
            M = (N - (npast + nfuture)) // stride + 1
            if M <= 0:
                continue

            block = {
                'X': torch.from_numpy(np.ascontiguousarray(X)),
                'Y': torch.from_numpy(np.ascontiguousarray(Y))
            }
            self.blocks.append(block)
            total += M
            self.cum_counts.append(total)

            if self.feature_dim is None: self.feature_dim = X.shape[1]
            if self.target_dim is None: self.target_dim = Y.shape[1]

        self.total = total

    def __len__(self):
        return self.total

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.total:
            raise IndexError('index out of range')

        block_idx = bisect_right(self.cum_counts, idx)
        prev = 0 if block_idx == 0 else self.cum_counts[block_idx - 1]
        offset = idx - prev
        start = offset * self.stride

        blk = self.blocks[block_idx]
        X_blk = blk['X']
        Y_blk = blk['Y']

        x_seq = X_blk[start:start + self.npast]
        y_seq = Y_blk[start + self.npast:start + self.npast + self.nfuture]
        return x_seq, y_seq

In [ ]:
# --- Dataset Builders ---

def build_quadriga_datasets(cfg: Cfg):
    paths = [cfg.mat_path] if isinstance(cfg.mat_path, str) else list(cfg.mat_path)
    csi_all, spd_all = [], []

    for p in tqdm(paths, desc='[data] MAT files', leave=False):
        try:
            with h5py.File(p, 'r') as f:
                csi_list = _load_h5_cell_array(f, 'csi_dataset')
                speed_list = _load_h5_cell_array(f, 'speed_dataset')
                m = min(len(csi_list), len(speed_list))
                csi_all.extend(csi_list[:m])
                spd_all.extend(speed_list[:m])
        except OSError as e:
            print(f"Error opening {p} (ensure -v7.3): {e}")
            continue

    train_runs, test_runs = [], []
    expected_in_dim, expected_out_dim = None, None

    for run_idx, (ci, spd) in enumerate(tqdm(zip(csi_all, spd_all), total=len(csi_all), desc='[data] runs', leave=False)):
        spd = np.asarray(spd).flatten().reshape(-1, 1).astype(np.float32)
        try:
            X_base = _to_H_time_major(ci)
        except Exception as e:
            continue
            
        if X_base.shape[0] <= 1: continue

        # Target: Delta H (Increment)
        Y_abs = X_base.copy()
        Y_delta = Y_abs[1:] - Y_abs[:-1]
        X_base = X_base[:-1]

        # Sync Speed
        spd = spd[:X_base.shape[0]]
        N_joint = min(X_base.shape[0], Y_delta.shape[0], spd.shape[0])
        if N_joint <= 0: continue
        
        X_base = X_base[:N_joint]
        Y_delta = Y_delta[:N_joint]
        spd = spd[:N_joint]

        # Features
        if cfg.use_speed_feature:
            X_feat = np.concatenate([X_base, spd], axis=1)
        else:
            X_feat = X_base

        # Normalization
        N = X_feat.shape[0]
        cut = int(cfg.train_frac * N)
        gap = (cfg.npast + cfg.nfuture - 1) if cfg.safety_gap else 0
        
        if cfg.norm_per_run:
            stats_src = X_feat[:cut] if cfg.norm_fit_train_only else X_feat
            Cchan = X_base.shape[1]
            
            mu_ch = stats_src[:, :Cchan].mean(axis=0, keepdims=True)
            sd_ch = stats_src[:, :Cchan].std(axis=0, keepdims=True) + 1e-8
            
            Xn_ch = (X_feat[:, :Cchan] - mu_ch) / sd_ch
            
            if cfg.use_speed_feature:
                sp = X_feat[:, Cchan:]
                mu_sp = sp.mean(axis=0, keepdims=True) if not cfg.norm_fit_train_only else sp[:cut].mean(axis=0, keepdims=True)
                sd_sp = sp.std(axis=0, keepdims=True) + 1e-8 if not cfg.norm_fit_train_only else sp[:cut].std(axis=0, keepdims=True) + 1e-8
                Xn = np.concatenate([Xn_ch, (sp - mu_sp) / sd_sp], axis=1)
            else:
                Xn = Xn_ch
                
            Yn = (Y_delta - mu_ch) / sd_ch
        else:
            Xn, Yn = X_feat, Y_delta

        # Check Dims
        if expected_in_dim is None:
            expected_in_dim = Xn.shape[1]
            expected_out_dim = Yn.shape[1]
        elif Xn.shape[1] != expected_in_dim:
            continue

        # Split
        req_len = cfg.npast + cfg.nfuture
        Xtr, Xte = Xn[:cut], Xn[cut+gap:]
        Ytr, Yte = Yn[:cut], Yn[cut+gap:]
        
        if Xtr.shape[0] >= req_len: train_runs.append((Xtr, Ytr))
        if Xte.shape[0] >= req_len: test_runs.append((Xte, Yte))

    if not train_runs:
        print("Warning: No valid runs found.")
        return None, None, 0, 0

    train_ds = QuaDRiGaRunWindows(train_runs, cfg.npast, cfg.nfuture, cfg.stride)
    test_ds = QuaDRiGaRunWindows(test_runs, cfg.npast, cfg.nfuture, cfg.stride)
    
    print(f"Dataset Built: {len(train_ds)} train windows, {len(test_ds)} test windows.")
    return train_ds, test_ds, expected_in_dim, expected_out_dim

def build_loaders(cfg: Cfg):
    train_ds, test_ds, dim_in, dim_out = build_quadriga_datasets(cfg)
    if train_ds is None: return None, None, 0, 0
    
    kw = {'num_workers': cfg.num_workers, 'pin_memory': (DEVICE=='cuda'), 'worker_init_fn': seed_worker}
    train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, **kw)
    val_dl = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, **kw)
    
    return train_dl, val_dl, dim_in, dim_out

In [ ]:
# --- Model Components ---

class DSLH(nn.Module):
    def __init__(self, Np: int, Nf: int, d_model: int, out_dim: int):
        """
        Dimension-wise Separable Linear Head (DSLH).
        Factorizes Projection into Time (Np->Nf) and Channel (d_model->out_dim).
        """
        super().__init__()
        self.W_time = nn.Parameter(torch.randn(Np, Nf) * (1.0 / math.sqrt(Np)))
        self.W_ch = nn.Linear(d_model, out_dim, bias=True)

    def forward(self, F: torch.Tensor) -> torch.Tensor:
        # F: [B, Np, d_model]
        Ft = F.transpose(1, 2)            # [B, d_model, Np]
        G = torch.matmul(Ft, self.W_time) # [B, d_model, Nf]
        G = G.transpose(1, 2)             # [B, Nf, d_model]
        Y = self.W_ch(G)                  # [B, Nf, out_dim]
        return Y

class Attention(nn.Module):
    """Global Luong (dot) attention."""
    def __init__(self, hidden_dim: int, attn_dropout: float = 0.0, scale: bool = False):
        super().__init__()
        self.drop = nn.Dropout(attn_dropout) if attn_dropout > 0 else nn.Identity()
        self.scale = 1.0 / math.sqrt(hidden_dim) if scale else 1.0

    def forward(self, enc_out: torch.Tensor, query: torch.Tensor):
        # enc_out: [B,T,H], query: [B,H]
        scores = torch.einsum("bth,bh->bt", enc_out, query)
        scores = scores * self.scale
        w = torch.softmax(scores, dim=-1)
        w = self.drop(w)
        context = torch.einsum("bt,bth->bh", w, enc_out)
        return context, w

# --- LinFormer Components ---
class TMLP(nn.Module):
    def __init__(self, N: int, d_model: int, dropout: float = 0.0):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(N, N) * (1.0 / math.sqrt(N)))
        self.W2 = nn.Parameter(torch.randn(N, N) * (1.0 / math.sqrt(N)))
        self.act = nn.ReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xt = x.transpose(1, 2)
        h1 = self.drop(self.act(torch.matmul(xt, self.W1)))
        h2 = torch.matmul(h1, self.W2).transpose(1, 2)
        return h2

class EncoderBlock(nn.Module):
    def __init__(self, N: int, d_model: int, ffn_mult: int = 1, dropout: float = 0.1):
        super().__init__()
        self.tmlp = TMLP(N, d_model, dropout)
        self.ln1 = nn.LayerNorm(d_model)
        hidden = ffn_mult * d_model
        self.ffn = nn.Sequential(
            nn.Linear(d_model, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_model), nn.Dropout(dropout)
        )
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.ln1(x + self.tmlp(x))
        x = self.ln2(x + self.ffn(x))
        return x

In [ ]:
# --- Proposed Model: GRU-Attn-DSLH ---

class GRUAttnBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim, cfg: Cfg, num_layers=1):
        super().__init__()
        self.bidirectional = cfg.bidirectional
        self.residual = cfg.gru_residual
        self.gated_fusion = cfg.gated_fusion
        
        self.rnn = nn.GRU(
            input_size=in_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=cfg.dropout if num_layers > 1 else 0.0,
            bidirectional=self.bidirectional
        )
        
        enc_dim = hidden_dim * (2 if self.bidirectional else 1)
        self.attn = Attention(enc_dim, attn_dropout=cfg.attn_dropout, scale=cfg.attn_scale)
        
        # Query Projection
        if cfg.query_proj:
            self.q_proj = nn.Sequential(nn.Linear(enc_dim, enc_dim), nn.LayerNorm(enc_dim))
        else:
            self.q_proj = nn.Identity()
            
        # Fusion
        fuse_dim = enc_dim * 2
        
        if self.gated_fusion:
            # Gated Fusion Mechanism
            self.gate_net = nn.Sequential(
                nn.Linear(fuse_dim, enc_dim // 4),
                nn.ReLU(),
                nn.Linear(enc_dim // 4, enc_dim)
            )
        else:
            self.cat_proj = nn.Linear(fuse_dim, enc_dim)
        
        self.ln_post = nn.LayerNorm(enc_dim)
        self.drop = nn.Dropout(cfg.dropout)
        self.res_proj = nn.Linear(in_dim, enc_dim) if (self.residual and in_dim != enc_dim) else nn.Identity()

    def forward(self, x):
        out, state = self.rnn(x)  # out: [B, T, enc_dim]
        
        # Prepare Query (last state)
        if self.bidirectional:
            raw_query = torch.cat([state[-2], state[-1]], dim=-1)
        else:
            raw_query = state[-1] # [B, H]
            
        query = self.q_proj(raw_query)
        ctx, w = self.attn(out, query) # ctx: [B, H]
        
        # Fusion
        B, T, H = out.shape
        ctx_exp = ctx.unsqueeze(1).expand(B, T, H)
        combined = torch.cat([out, ctx_exp], dim=-1) # [B, T, 2H]
        
        if self.gated_fusion:
            z = torch.sigmoid(self.gate_net(combined))
            fused = z * out + (1 - z) * ctx_exp
        else:
            fused = self.cat_proj(combined)
            
        fused = self.ln_post(fused)
        fused = self.drop(fused)
        
        if self.residual:
            fused = fused + self.res_proj(x)
            
        return fused

class GRUAttnSeq(nn.Module):
    def __init__(self, in_dim, out_dim, Np, Nf, cfg: Cfg):
        super().__init__()
        self.d_model = getattr(cfg, "d_model_rnn", cfg.d_model)
        self.embed = nn.Linear(in_dim, self.d_model)
        
        self.blocks = nn.ModuleList()
        # Single block with N layers internal to GRU usually, or stacked blocks
        if cfg.stacked_gru:
            for _ in range(cfg.n_layers):
                self.blocks.append(GRUAttnBlock(self.d_model, self.d_model, cfg, num_layers=1))
        else:
            self.blocks.append(GRUAttnBlock(self.d_model, self.d_model, cfg, num_layers=cfg.n_layers))
            
        final_dim = self.d_model * (2 if cfg.bidirectional else 1)
        self.head = DSLH(Np, Nf, final_dim, out_dim)

    def forward(self, x):
        h = self.embed(x)
        for blk in self.blocks:
            h = blk(h)
        return self.head(h)

In [ ]:
# --- Baseline Model: LinFormer ---

class LinFormer(nn.Module):
    def __init__(self, in_dim, out_dim, Np, Nf, d_model=512, n_layers=6, ffn_mult=1, dropout=0.1):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        self.blocks = nn.ModuleList([
            EncoderBlock(Np, d_model, ffn_mult, dropout) for _ in range(n_layers)
        ])
        self.head = DSLH(Np, Nf, d_model, out_dim)

    def forward(self, x):
        h = self.embed(x) # [B, Np, d_model]
        for blk in self.blocks:
            h = blk(h)
        return self.head(h)

In [ ]:
# --- Baseline Models: Vanilla LSTM & Vanilla GRU ---

class VanillaLSTM(nn.Module):
    """
    Standard LSTM baseline for CSI prediction.
    Uses a multi-layer LSTM encoder followed by the DSLH prediction head.
    No attention or gated fusion — pure sequential modelling.
    """
    def __init__(self, in_dim, out_dim, Np, Nf, d_model=256, n_layers=3, dropout=0.3):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.ln = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.head = DSLH(Np, Nf, d_model, out_dim)

    def forward(self, x):
        h = self.embed(x)                # [B, Np, d_model]
        out, _ = self.lstm(h)            # [B, Np, d_model]
        out = self.drop(self.ln(out))    # LayerNorm + Dropout
        return self.head(out)            # [B, Nf, out_dim]


class VanillaGRU(nn.Module):
    """
    Standard GRU baseline for CSI prediction.
    Uses a multi-layer GRU encoder followed by the DSLH prediction head.
    No attention or gated fusion — pure sequential modelling.
    """
    def __init__(self, in_dim, out_dim, Np, Nf, d_model=256, n_layers=3, dropout=0.3):
        super().__init__()
        self.embed = nn.Linear(in_dim, d_model)
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.ln = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
        self.head = DSLH(Np, Nf, d_model, out_dim)

    def forward(self, x):
        h = self.embed(x)                # [B, Np, d_model]
        out, _ = self.gru(h)             # [B, Np, d_model]
        out = self.drop(self.ln(out))    # LayerNorm + Dropout
        return self.head(out)            # [B, Nf, out_dim]

In [ ]:
# --- Loss & Helper Metrics ---

class WMSELoss(nn.Module):
    def __init__(self, Nf: int, l2_lambda: float = 0.0):
        super().__init__()
        # Weighted MSE: earlier steps matter more (decaying weights)
        self.register_buffer('w', (torch.arange(1, Nf+1, dtype=torch.float32) ** -0.5))
        self.l2_lambda = l2_lambda

    def forward(self, pred, target):
        diff2 = (pred - target) ** 2
        w = self.w.view(1, -1, 1)
        loss = (w * diff2).mean(dim=(1,2))
        
        if self.l2_lambda > 0:
            l2 = (pred ** 2).mean(dim=(1,2))
            loss = loss + self.l2_lambda * l2
            
        return loss.mean()

def nmse_db(pred, target):
    num = torch.sum((pred - target)**2)
    den = torch.sum(target**2) + 1e-12
    return float((10.0 * torch.log10(num / den)).detach().cpu())

# --- Evaluation Loop ---
@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval()
    tot_loss = 0.0
    num_accum = 0.0
    den_accum = 0.0
    total_samples = 0
    
    for X, Y in loader:
        X = X.to(DEVICE); Y = Y.to(DEVICE)
        B = X.size(0)
        P = model(X)
        loss = loss_fn(P, Y)
        tot_loss += float(loss.detach().cpu()) * B
        
        num_accum += torch.sum((P - Y)**2).item()
        den_accum += torch.sum(Y**2).item()
        total_samples += B
        
    avg_loss = tot_loss / max(total_samples, 1)
    nmse = 10.0 * np.log10(num_accum / (den_accum + 1e-12))
    return avg_loss, nmse

# --- Training Loop ---
def train_model(model, train_dl, val_dl, cfg: Cfg, tag: str):
    set_global_seed(cfg.seed)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = WMSELoss(cfg.nfuture, l2_lambda=cfg.pred_l2_lambda).to(DEVICE)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    # Scheduler
    warmup_epochs = min(5, max(2, cfg.epochs // 4))
    def lr_lambda(ep):
        if ep < warmup_epochs: return (ep + 1) / warmup_epochs
        progress = (ep - warmup_epochs) / max(1, cfg.epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    best_val = float('inf')
    best_nmse = float('inf')
    epochs_no_improve = 0
    
    print(f"\nStarting training for {tag}...")
    
    model.train()
    for epoch in range(1, cfg.epochs+1):
        run_loss = 0.0
        steps = 0
        
        pbar = tqdm(train_dl, desc=f"Ep {epoch:02d}", leave=False)
        for X, Y in pbar:
            X = X.to(DEVICE); Y = Y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            
            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                P = model(X)
                loss = loss_fn(P, Y)
                
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            
            run_loss += loss.item() * X.size(0)
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")
            
        train_loss = run_loss / len(train_dl.dataset)
        val_loss, val_nmse = evaluate(model, val_dl, loss_fn)
        scheduler.step()
        
        print(f"[{tag}] Ep {epoch:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | NMSE: {val_nmse:.2f} dB")
        
        # Save Best
        if val_loss < best_val:
            best_val = val_loss
            best_nmse = val_nmse
            torch.save(model.state_dict(), os.path.join(cfg.out_dir, f'best_{tag}.pt'))
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if cfg.early_stopping and epochs_no_improve >= cfg.early_patience:
                print(f"Early stopping at epoch {epoch}")
                break
                
    return best_val, best_nmse

In [ ]:
# --- Check Params ---
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# --- Main Execution Block ---

# 1. Build Data
train_dl, val_dl, IN_DIM, OUT_DIM = build_loaders(cfg)

if train_dl:
    print(f"\nData Ready. In Dim: {IN_DIM}, Out Dim: {OUT_DIM}")

    # 2. Instantiate Models
    models = {}
    
    # Proposed
    models['GRU-Attn-DSLH'] = GRUAttnSeq(IN_DIM, OUT_DIM, cfg.npast, cfg.nfuture, cfg).to(DEVICE)
    
    # Baselines
    models['Vanilla-LSTM'] = VanillaLSTM(
        IN_DIM, OUT_DIM, cfg.npast, cfg.nfuture,
        d_model=cfg.d_model_rnn,
        n_layers=cfg.n_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    models['Vanilla-GRU'] = VanillaGRU(
        IN_DIM, OUT_DIM, cfg.npast, cfg.nfuture,
        d_model=cfg.d_model_rnn,
        n_layers=cfg.n_layers,
        dropout=cfg.dropout
    ).to(DEVICE)

    models['LinFormer'] = LinFormer(
        IN_DIM, OUT_DIM, cfg.npast, cfg.nfuture, 
        d_model=cfg.d_model_linformer, 
        n_layers=cfg.n_layers, 
        ffn_mult=cfg.ffn_mult, 
        dropout=cfg.dropout
    ).to(DEVICE)

    # 3. Print Complexity
    # print("\n--- Model Complexity ---")
    # for name, m in models.items():
    #     print(f"{name}: {count_params(m)/1e6:.3f} M params")

    # 4. Train
    results = []
    for name, model in models.items():
        vloss, vnmse = train_model(model, train_dl, val_dl, cfg, tag=name)
        results.append({'model': name, 'val_loss': vloss, 'val_nmse': vnmse})
        
    print("\n--- Final Results ---")
    for r in results:
        print(f"{r['model']}: NMSE = {r['val_nmse']:.2f} dB")
else:
    print("\nData loading failed or returned no data. Check `cfg.mat_path`.")